<a href="https://colab.research.google.com/github/ananthakrishnanm010/Real_or_Fake/blob/main/FakeVsReal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Libraries


In [ ]:
import pandas as pd
import numpy as np
import string
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from sklearn.metrics import classification_report, confusion_matrix


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


#Read dataset

In [ ]:
df_fake = pd.read_csv('/content/drive/MyDrive/ICT_Data_Science/Fake_real/Fake.csv')
display(df_fake.head())

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [ ]:
df_true = pd.read_csv('/content/drive/MyDrive/ICT_Data_Science/Fake_real/True.csv')
display(df_true.head())

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [ ]:
df_true["label"] = 1
df_fake["label"] = 0

In [ ]:
df = pd.concat([df_true, df_fake], ignore_index=True)


In [ ]:
df

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1
...,...,...,...,...,...
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0


# EDA

In [ ]:
df_fake.head(5)

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [ ]:
df_fake.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
 4   label    23481 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 917.4+ KB


In [ ]:
df.shape

(44898, 5)

In [ ]:
df_true.size

107085

In [ ]:
df_fake.size

117405

In [ ]:
df.describe()

,label
count,44898.000000
mean,0.477015
std,0.499477
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


In [ ]:
df.isnull().sum()

,0
title,0
text,0
subject,0
date,0
label,0


## Text Preprocessing

In [ ]:
# Create a separate DataFrame for the cleaned text output
df_clean_text = df.copy()

### Duplicate Removal

In [ ]:
# Check for duplicate entries
df.duplicated().sum()

np.int64(209)

In [ ]:
# Remove duplicate entries
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
df_clean_text = df.copy()


In [ ]:
df.duplicated().sum()

np.int64(0)

### Punctuation Removal

In [ ]:
# Function to remove punctuatio
def remove_punc(text):
  text = str(text)
  text = text.replace("�", "")
  punctuationless_text = ''.join(i for i in text if i not in string.punctuation)
  return punctuationless_text

In [ ]:
df["punctuationless_text"] = [remove_punc(text) for text in df["text"]]

In [ ]:
df

,title,text,subject,date,label,punctuationless_text
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The head of a conservative...
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,WASHINGTON Reuters Transgender people will be...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The special counsel invest...
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1,WASHINGTON Reuters Trump campaign adviser Geo...
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1,SEATTLEWASHINGTON Reuters President Donald Tr...
...,...,...,...,...,...,...
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0,21st Century Wire says As 21WIRE reported earl...
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0,21st Century Wire says It s a familiar theme W...
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0,Patrick Henningsen 21st Century WireRemember ...
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0,21st Century Wire says Al Jazeera America will...


### Lowercasing

In [ ]:
# Convert text to lowercase
df["lowercased"] = [text.lower() for text in df["punctuationless_text"]]

In [ ]:
df.head(3)

,title,text,subject,date,label,punctuationless_text,lowercased
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The head of a conservative...,washington reuters the head of a conservative...
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,WASHINGTON Reuters Transgender people will be...,washington reuters transgender people will be...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The special counsel invest...,washington reuters the special counsel invest...


### Tokenization

In [ ]:
# we convert the entire text into a list of unique words.
def tokenization(text):
  word_list = nltk.word_tokenize(text)
  return word_list


In [ ]:
df["tokenized"] = [tokenization(text) for text in df["lowercased"]]

In [ ]:
df.head(3)

,title,text,subject,date,label,punctuationless_text,lowercased,tokenized
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The head of a conservative...,washington reuters the head of a conservative...,"[washington, reuters, the, head, of, a, conser..."
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,WASHINGTON Reuters Transgender people will be...,washington reuters transgender people will be...,"[washington, reuters, transgender, people, wil..."
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The special counsel invest...,washington reuters the special counsel invest...,"[washington, reuters, the, special, counsel, i..."


### Stop words removal

In [ ]:
# Remove stop words
stop_words_list = nltk.corpus.stopwords.words('english')
stop_words_list
def remove_stopwords(tokens):
  stop_word_less_list = [i for i in tokens if i not in stop_words_list]
  return stop_word_less_list

In [ ]:
df["stop_words_removed"] = [remove_stopwords(text) for text in df["tokenized"]]

In [ ]:
df.head(3)

,title,text,subject,date,label,punctuationless_text,lowercased,tokenized,stop_words_removed
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The head of a conservative...,washington reuters the head of a conservative...,"[washington, reuters, the, head, of, a, conser...","[washington, reuters, head, conservative, repu..."
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,WASHINGTON Reuters Transgender people will be...,washington reuters transgender people will be...,"[washington, reuters, transgender, people, wil...","[washington, reuters, transgender, people, all..."
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The special counsel invest...,washington reuters the special counsel invest...,"[washington, reuters, the, special, counsel, i...","[washington, reuters, special, counsel, invest..."


### Lemmatization

In [ ]:
lem_obj = WordNetLemmatizer()
def lemmatizing(stem_token_list):
  lem_list = [lem_obj.lemmatize(word) for word in stem_token_list ]
  return lem_list

In [ ]:
df["lemmatized_text"] = [lemmatizing(text) for text in df["stop_words_removed"]]

In [ ]:
df.head(3)

,title,text,subject,date,label,punctuationless_text,lowercased,tokenized,stop_words_removed,lemmatized_text
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The head of a conservative...,washington reuters the head of a conservative...,"[washington, reuters, the, head, of, a, conser...","[washington, reuters, head, conservative, repu...","[washington, reuters, head, conservative, repu..."
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,WASHINGTON Reuters Transgender people will be...,washington reuters transgender people will be...,"[washington, reuters, transgender, people, wil...","[washington, reuters, transgender, people, all...","[washington, reuters, transgender, people, all..."
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The special counsel invest...,washington reuters the special counsel invest...,"[washington, reuters, the, special, counsel, i...","[washington, reuters, special, counsel, invest...","[washington, reuters, special, counsel, invest..."


In [ ]:
df_clean_text['text'] = df['lemmatized_text'].apply(lambda x: ' '.join(x))

In [ ]:
df_clean_text.head()

,title,text,subject,date,label,punctuationless_text,lowercased,tokenized,stop_words_removed,lemmatized_text
0,"As U.S. budget fight looms, Republicans flip t...",washington reuters head conservative republica...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The head of a conservative...,washington reuters the head of a conservative...,"[washington, reuters, the, head, of, a, conser...","[washington, reuters, head, conservative, repu...","[washington, reuters, head, conservative, repu..."
1,U.S. military to accept transgender recruits o...,washington reuters transgender people allowed ...,politicsNews,"December 29, 2017",1,WASHINGTON Reuters Transgender people will be...,washington reuters transgender people will be...,"[washington, reuters, transgender, people, wil...","[washington, reuters, transgender, people, all...","[washington, reuters, transgender, people, all..."
2,Senior U.S. Republican senator: 'Let Mr. Muell...,washington reuters special counsel investigati...,politicsNews,"December 31, 2017",1,WASHINGTON Reuters The special counsel invest...,washington reuters the special counsel invest...,"[washington, reuters, the, special, counsel, i...","[washington, reuters, special, counsel, invest...","[washington, reuters, special, counsel, invest..."
3,FBI Russia probe helped by Australian diplomat...,washington reuters trump campaign adviser geor...,politicsNews,"December 30, 2017",1,WASHINGTON Reuters Trump campaign adviser Geo...,washington reuters trump campaign adviser geo...,"[washington, reuters, trump, campaign, adviser...","[washington, reuters, trump, campaign, adviser...","[washington, reuters, trump, campaign, adviser..."
4,Trump wants Postal Service to charge 'much mor...,seattlewashington reuters president donald tru...,politicsNews,"December 29, 2017",1,SEATTLEWASHINGTON Reuters President Donald Tr...,seattlewashington reuters president donald tr...,"[seattlewashington, reuters, president, donald...","[seattlewashington, reuters, president, donald...","[seattlewashington, reuters, president, donald..."


# Embedding

## TF-IDF Implementation

In [ ]:
# Initialize TF-IDF Vectorizer
# The vectorizer is fitted after the train/test split to avoid data leakage.
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Ensure the cleaned text is a string
df['text'] = df['text'].fillna('').astype(str)

print("Number of text samples:", len(df))


Shape of TF-IDF vectors: (44689, 5000)


## Incorporating Date and Subject Features

### Processing 'date' column

In [ ]:
# Convert 'date' to datetime and extract features
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek # Monday=0, Sunday=6

# Drop original 'date' column
df = df.drop(columns=['date'])

display(df[['year', 'month', 'day_of_week']].head())

### Processing 'subject' column using One-Hot Encoding

In [ ]:
# Initialize OneHotEncoder for the subject feature.
# It will be fitted only on the training data to avoid data leakage.
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    # Compatibility with older scikit-learn versions
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=True)

print("OneHotEncoder initialized.")


### Combining all features

In [ ]:
# Prepare the additional structured features.
# Missing date-derived values are replaced with 0.
numeric_features = df[['year', 'month', 'day_of_week']].fillna(0).astype(float)

# Keep subject as a DataFrame so the encoder can be fitted after the train/test split.
subject_features = df[['subject']].fillna('Unknown').astype(str)

print("Numeric feature shape:", numeric_features.shape)
print("Subject feature shape:", subject_features.shape)


## Data Splitting

In [ ]:
# Define target
y = df['label'].astype(int)

# Split row indices first. This lets us fit TF-IDF and OneHotEncoder
# only on the training portion and prevents data leakage.
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Text features: fit only on training text
X_text_train = df.iloc[train_idx]['text']
X_text_test = df.iloc[test_idx]['text']

X_tfidf_train = tfidf_vectorizer.fit_transform(X_text_train)
X_tfidf_test = tfidf_vectorizer.transform(X_text_test)

# Subject features: fit only on training subjects
X_subject_train = ohe.fit_transform(subject_features.iloc[train_idx])
X_subject_test = ohe.transform(subject_features.iloc[test_idx])

# Date features
X_date_train = csr_matrix(numeric_features.iloc[train_idx].values)
X_date_test = csr_matrix(numeric_features.iloc[test_idx].values)

# Combine TF-IDF + date + subject features
X_train = hstack([X_tfidf_train, X_date_train, X_subject_train]).tocsr()
X_test = hstack([X_tfidf_test, X_date_test, X_subject_test]).tocsr()

y_train = y.iloc[train_idx].to_numpy()
y_test = y.iloc[test_idx].to_numpy()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


## Model Training and Evaluation

### Model 1: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize and train the Logistic Regression model
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
log_reg_model.fit(X_train, y_train)

# Make predictions
y_pred_log_reg = log_reg_model.predict(X_test)

# Evaluate the model
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_log_reg))

# Confusion Matrix
cm_log_reg = confusion_matrix(y_test, y_pred_log_reg)
sns.heatmap(cm_log_reg, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake', 'True'], yticklabels=['Fake', 'True'])
plt.title('Logistic Regression Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Model 2: Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt

# Initialize and train the Decision Tree model
dec_tree_model = DecisionTreeClassifier(random_state=42)
dec_tree_model.fit(X_train, y_train)

# Make predictions
y_pred_dec_tree = dec_tree_model.predict(X_test)

# Evaluate the model
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dec_tree))

# Confusion Matrix
cm_dec_tree = confusion_matrix(y_test, y_pred_dec_tree)
sns.heatmap(cm_dec_tree, annot=True, fmt='d', cmap='Greens', xticklabels=['Fake', 'True'], yticklabels=['Fake', 'True'])
plt.title('Decision Tree Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Model 3: Neural Network (TensorFlow/Keras)

In [ ]:
# Convert sparse matrix to dense for TensorFlow
# A dense representation is required for this neural-network architecture.
X_train_dense = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)
X_test_dense = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)

# Build the Neural Network model
nn_model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train_dense.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# Compile the model
nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

nn_model.summary()

# Train the model
history = nn_model.fit(
    X_train_dense,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# Evaluate the model
loss, accuracy = nn_model.evaluate(X_test_dense, y_test, verbose=0)
print(f"\nNeural Network Test Accuracy: {accuracy:.4f}")

# Make predictions
y_pred_nn_proba = nn_model.predict(X_test_dense, verbose=0).ravel()
y_pred_nn = (y_pred_nn_proba >= 0.5).astype(int)

print("\nNeural Network Classification Report:")
print(classification_report(y_test, y_pred_nn))

cm_nn = confusion_matrix(y_test, y_pred_nn)
sns.heatmap(
    cm_nn,
    annot=True,
    fmt='d',
    cmap='Purples',
    xticklabels=['Fake', 'True'],
    yticklabels=['Fake', 'True']
)
plt.title('Neural Network Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()
